# 🥈 Notebook 02 — Silver Layer: Build Dimension Tables

**Goal:** Clean Bronze data and build 5 dimension tables — the backbone of the star schema.

> **Run time:** ~5 min

```
Bronze Tables
  └──► dim_customer   — Who     (500 rows)
  └──► dim_account    — What    (500 rows)
  └──► dim_product    — Product (10  rows)
  └──► dim_branch     — Where   (10  rows)
  └──► dim_date       — When    (auto-generated, 2020-2025)
```

In [ ]:
# Import Spark SQL functions used to derive silver-dimension attributes.
from pyspark.sql import functions as F

# Announce the silver layer build so notebook output clearly shows the current step.
print('Building Silver dimension tables...')

## dim_customer

In [ ]:
# Build dim_customer from bronze_customers by removing ingestion metadata and deduplicating CustomerID.
# Add analytical attributes such as FullName, Age, AgeGroup, CreditScoreTier, and JoinYear for reporting.
# Stamp the customer dimension refresh time before publishing the curated silver table.
dim_customer = spark.table('bronze_customers') \
    .drop('_ingested_at','_source_file') \
    .dropDuplicates(['CustomerID']) \
    .withColumn('FullName', F.concat_ws(' ', 'FirstName', 'LastName')) \
    .withColumn('Age', F.floor(F.months_between(F.current_date(), F.to_date('DateOfBirth','yyyy-MM-dd')) / 12)) \
    .withColumn('AgeGroup',
        F.when(F.col('Age') < 30, 'Under 30')
         .when(F.col('Age') < 45, '30-44')
         .when(F.col('Age') < 60, '45-59')
         .otherwise('60+')) \
    .withColumn('CreditScoreTier',
        F.when(F.col('CreditScore') >= 750, 'Excellent')
         .when(F.col('CreditScore') >= 670, 'Good')
         .when(F.col('CreditScore') >= 580, 'Fair')
         .otherwise('Poor')) \
    .withColumn('JoinYear', F.year(F.to_date('JoinDate','yyyy-MM-dd'))) \
    .withColumn('_updated_at', F.current_timestamp())

# Write dim_customer to Delta so downstream fact builds can reuse a conformed customer dimension.
dim_customer.write.format('delta').mode('overwrite').saveAsTable('dim_customer')
# Print the customer dimension row count as a quick load validation.
print(f'dim_customer: {dim_customer.count()} rows')
# Preview customer segmentation columns that will feed the gold star schema.
dim_customer.select('CustomerID','FullName','CreditScoreTier','AgeGroup','CustomerSegment').show(5)

## dim_account

In [ ]:
# Build dim_account from bronze_accounts by removing ingestion metadata and deduplicating AccountID.
# Derive OpenYear and BalanceTier so account tenure and balance bands are available for analytics.
# Stamp the account dimension refresh time before saving the curated silver table.
dim_account = spark.table('bronze_accounts') \
    .drop('_ingested_at','_source_file') \
    .dropDuplicates(['AccountID']) \
    .withColumn('OpenYear', F.year(F.to_date('OpenDate','yyyy-MM-dd'))) \
    .withColumn('BalanceTier',
        F.when(F.col('Balance') >= 100000, 'High Value')
         .when(F.col('Balance') >= 25000,  'Mid Value')
         .when(F.col('Balance') >= 5000,   'Standard')
         .otherwise('Low Balance')) \
    .withColumn('_updated_at', F.current_timestamp())

# Write dim_account to Delta for reuse in downstream fact table joins.
dim_account.write.format('delta').mode('overwrite').saveAsTable('dim_account')
# Print the account dimension row count to verify the silver load completed.
print(f'dim_account: {dim_account.count()} rows')
# Preview the account dimension to confirm the derived balance bands look correct.
dim_account.show(5)

## dim_product

In [ ]:
# Build dim_product from bronze_products by removing ingestion metadata and deduplicating ProductID.
# Calculate RateSpread so each product exposes the difference between its minimum and maximum rates.
# Stamp the product dimension refresh time before publishing the silver table.
dim_product = spark.table('bronze_products') \
    .drop('_ingested_at','_source_file') \
    .dropDuplicates(['ProductID']) \
    .withColumn('RateSpread', F.round(F.col('MaxInterestRate') - F.col('MinInterestRate'), 2)) \
    .withColumn('_updated_at', F.current_timestamp())

# Write dim_product to Delta so loan facts can join to product attributes.
dim_product.write.format('delta').mode('overwrite').saveAsTable('dim_product')
# Print the product dimension row count to validate the load.
print(f'dim_product: {dim_product.count()} rows')
# Preview the product dimension to confirm product and pricing attributes were derived correctly.
dim_product.show()

## dim_branch

In [ ]:
# Build dim_branch from bronze_branches by removing ingestion metadata and deduplicating BranchID.
# Stamp the branch dimension refresh time before saving the curated silver table.
dim_branch = spark.table('bronze_branches') \
    .drop('_ingested_at','_source_file') \
    .dropDuplicates(['BranchID']) \
    .withColumn('_updated_at', F.current_timestamp())

# Write dim_branch to Delta so branch and region attributes can be reused downstream.
dim_branch.write.format('delta').mode('overwrite').saveAsTable('dim_branch')
# Print the branch dimension row count to confirm the load completed.
print(f'dim_branch: {dim_branch.count()} rows')
# Preview the branch dimension to verify branch metadata is ready for fact joins.
dim_branch.show()

## dim_date — Time Intelligence

In [ ]:
# Generate every date from 2020-01-01 to 2025-12-31
# Generate dim_date by expanding a calendar range into one row per day for time-based analysis.
# Derive calendar and fiscal attributes such as Year, Quarter, Month, Week, DayName, and IsWeekend.
dim_date = spark.sql("SELECT explode(sequence(to_date('2020-01-01'), to_date('2025-12-31'), interval 1 day)) AS Date") \
    .withColumn('DateKey',      F.date_format('Date','yyyyMMdd').cast('int')) \
    .withColumn('Year',         F.year('Date')) \
    .withColumn('Quarter',      F.quarter('Date')) \
    .withColumn('QuarterName',  F.concat(F.lit('Q'), F.quarter('Date').cast('string'))) \
    .withColumn('Month',        F.month('Date')) \
    .withColumn('MonthName',    F.date_format('Date','MMMM')) \
    .withColumn('MonthShort',   F.date_format('Date','MMM')) \
    .withColumn('Week',         F.weekofyear('Date')) \
    .withColumn('DayOfMonth',   F.dayofmonth('Date')) \
    .withColumn('DayOfWeek',    F.dayofweek('Date')) \
    .withColumn('DayName',      F.date_format('Date','EEEE')) \
    .withColumn('IsWeekend',    F.when(F.dayofweek('Date').isin([1,7]), True).otherwise(False)) \
    .withColumn('FiscalYear',   F.when(F.month('Date') >= 10, F.year('Date') + 1).otherwise(F.year('Date'))) \
    .withColumn('FiscalQuarter',F.when(F.month('Date').isin([10,11,12]), 1)
                                 .when(F.month('Date').isin([1,2,3]), 2)
                                 .when(F.month('Date').isin([4,5,6]), 3)
                                 .otherwise(4))

# Write dim_date to Delta so every fact table can resolve transaction and loan dates consistently.
dim_date.write.format('delta').mode('overwrite').saveAsTable('dim_date')
# Print the date dimension row count to confirm the full calendar range was created.
print(f'dim_date: {dim_date.count()} rows (2020-2025)')
# Preview the date dimension to validate the generated calendar attributes.
dim_date.show(3)

## Silver Summary + Quality Checks

In [ ]:
%%sql
-- Compare row counts across all silver dimensions after the curation step completes.
SELECT 'dim_customer' AS DimTable, COUNT(*) AS Rows FROM dim_customer UNION ALL
SELECT 'dim_account',               COUNT(*)         FROM dim_account  UNION ALL
SELECT 'dim_product',               COUNT(*)         FROM dim_product  UNION ALL
SELECT 'dim_branch',                COUNT(*)         FROM dim_branch   UNION ALL
SELECT 'dim_date',                  COUNT(*)         FROM dim_date
ORDER BY Rows DESC

In [ ]:
%%sql
-- Verify that every account in dim_account resolves to a valid customer in dim_customer.
-- Referential integrity check: accounts must have valid customers
SELECT 'Orphan Accounts' AS Check, COUNT(*) AS Count
FROM dim_account a
LEFT JOIN dim_customer c ON a.CustomerID = c.CustomerID
WHERE c.CustomerID IS NULL
UNION ALL
SELECT 'Valid Accounts', COUNT(*)
FROM dim_account a
INNER JOIN dim_customer c ON a.CustomerID = c.CustomerID